# Notebook 07 — Cross-Model Comparative Evaluation

**Author:** Kaveesha Athukorala (Member 4)
**Module:** SE4050 — Deep Learning (2026)

This notebook is the **aggregator** for the four-architecture Food-101 benchmark. It reads only
committed artifacts — it trains nothing and needs neither a GPU nor the dataset, so it reproduces
end-to-end on any machine with `pandas` and `matplotlib`.

**Reads** `results/<model>/metrics.json`, `history.csv` and `classification_report.json` for
`custom_cnn`, `mobilenetv2`, `resnet50` and `efficientnetb0`.

**Writes**

| Output | Purpose |
| :--- | :--- |
| `results/comparison/master_comparison.csv` | machine-readable master table |
| `results/comparison/master_comparison.md` | Section 7 master table, paste-ready |
| `results/comparison/per_class_f1_all_models.csv` | all 101 classes x 4 models |
| `results/comparison/hardest_classes.csv` | the ten hardest classes |
| `report/figures/fig6_comparative_learning_curves.png` | Section 7 learning curves |
| `report/figures/fig7_comparative_metrics.png` | Section 7 metric panels |
| `report/figures/fig8_accuracy_vs_parameters.png` | Section 8 accuracy-vs-complexity |
| `report/figures/fig9_hardest_classes.png` | Section 8 shared error clusters |

In [ ]:
# Environment sync and imports.
# Unlike notebooks 03-06 this one needs no GPU and no dataset - only committed artifacts.

import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

REPO_NAME = "food-classification-deep-learning"
BRANCH = "feature/kaveesha-efficientnetb0"

if 'google.colab' in sys.modules:
    if not os.path.exists(f"/content/{REPO_NAME}"):
        !git clone -b {BRANCH} https://github.com/niRmana11/food-classification-deep-learning.git
        %cd /content/{REPO_NAME}
    else:
        %cd /content/{REPO_NAME}
        !git checkout {BRANCH}
        !git pull origin {BRANCH}
    if f"/content/{REPO_NAME}" not in sys.path:
        sys.path.insert(0, f"/content/{REPO_NAME}")

ROOT = Path.cwd() if (Path.cwd() / "results").exists() else Path.cwd().parent
RESULTS = ROOT / "results"
COMPARISON = RESULTS / "comparison"
FIGURES = ROOT / "report" / "figures"
COMPARISON.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

MODELS = ["custom_cnn", "mobilenetv2", "resnet50", "efficientnetb0"]
DISPLAY = {"custom_cnn": "Custom CNN", "mobilenetv2": "MobileNetV2",
           "resnet50": "ResNet-50", "efficientnetb0": "EfficientNetB0"}
SCALING = {"custom_cnn": "None (from scratch)", "mobilenetv2": "Width (depthwise)",
           "resnet50": "Depth (residual)", "efficientnetb0": "Compound (d-w-r)"}

# Categorical palette: slots 1, 2, 3 and 7 of the reference data-viz palette.
# Slot 4 (yellow) was swapped for violet because the yellow/orange pair fails the
# all-pairs normal-vision separation floor; this four-colour set passes every check
# (lightness band, chroma floor, CVD separation, normal-vision floor) on a white surface.
# Colour is bound to the ENTITY and never to rank, so a model keeps its hue in every figure.
COLOR = {"efficientnetb0": "#2a78d6",  # blue
         "resnet50":       "#eb6834",  # orange
         "mobilenetv2":    "#1baf7a",  # aqua
         "custom_cnn":     "#4a3aa7"}  # violet
INK, INK_2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"

plt.rcParams.update({
    "figure.dpi": 140, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica Neue", "Helvetica", "Arial", "DejaVu Sans"],
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.titleweight": "semibold", "axes.titlecolor": INK, "axes.labelcolor": INK_2,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "grid.color": GRID, "grid.linewidth": 0.8,
    "legend.frameon": False, "legend.fontsize": 8,
    "figure.facecolor": "white", "axes.facecolor": "white",
})

print(f"[INFO] Repository root: {ROOT}")
print(f"[INFO] matplotlib {matplotlib.__version__} | pandas {pd.__version__}")

## 7.1 Schema normalisation

The four `metrics.json` files were written independently and **do not share a schema**. Three
divergences have to be reconciled before any table can be built, and each is handled explicitly
rather than silently:

1. **Key naming.** ResNet-50 and MobileNetV2 record `test_top1_accuracy`; Custom CNN and
   EfficientNetB0 record `final_test_accuracy`. Both spellings are accepted.
2. **Missing macro metrics.** Custom CNN and EfficientNetB0 omit macro precision/recall/F1 from
   `metrics.json`. These are recovered from the `macro avg` block of each
   `classification_report.json`, so no model is left blank.
3. **Incomparable size figures.** `model_size_mb` means different things per model — EfficientNetB0
   logged a `.keras` checkpoint that also serialises the Adam moment tensors (41.47 MB against a
   true 15.96 MiB of weights), while ResNet-50 logged 98.40 MB against 90.80 MiB of weights.
   A single uniform basis, `total_params x 4 bytes` (exactly what Keras prints in `model.summary()`),
   is therefore computed for all four. The originally reported values are retained in a separate
   column and are **not** used for comparison.

Custom CNN's **test** Top-5 was never recorded and cannot be recovered without re-evaluating the
model, so it is reported as missing rather than imputed. Its **validation** Top-5 is recoverable
from the last row of `history.csv`.

In [ ]:
def weights_mib(total_params: int) -> float:
    """Float32 weight footprint - the uniform basis Keras itself reports in model.summary()."""
    return total_params * 4 / (1024 ** 2)


rows, histories, reports = [], {}, {}

for m in MODELS:
    d = RESULTS / m
    met = json.loads((d / "metrics.json").read_text())
    reports[m] = json.loads((d / "classification_report.json").read_text())
    histories[m] = pd.read_csv(d / "history.csv")
    macro = reports[m].get("macro avg", {})
    last = histories[m].iloc[-1]

    def pick(*keys):
        """First non-null value among competing key spellings."""
        for k in keys:
            if met.get(k) is not None:
                return met[k]
        return np.nan

    val_top5 = pick("val_top5_accuracy", "final_val_top5_accuracy")
    if np.isnan(val_top5):                       # recover from the training log
        val_top5 = float(last.get("val_top_5_accuracy", np.nan))

    rows.append({
        "model": DISPLAY[m], "key": m, "scaling": SCALING[m],
        "total_params": met["total_parameters"],
        "trainable_params": met["trainable_parameters"],
        "weights_mib": weights_mib(met["total_parameters"]),
        "reported_size_mb": met.get("model_size_mb", np.nan),   # NOT comparable - see markdown
        "train_time_s": met["training_time_seconds"],
        "epochs": int(len(histories[m])),
        "val_top1": pick("val_top1_accuracy", "final_val_accuracy"),
        "val_top5": val_top5,
        "test_top1": pick("test_top1_accuracy", "final_test_accuracy"),
        "test_top5": pick("test_top5_accuracy", "final_test_top5_accuracy"),
        "test_loss": pick("final_test_loss", "test_loss"),
        "val_loss": pick("final_val_loss", "val_loss"),
        "macro_precision": macro.get("precision", np.nan),
        "macro_recall": macro.get("recall", np.nan),
        "macro_f1": macro.get("f1-score", np.nan),
        "final_train_acc": float(last["accuracy"]),
        "final_val_acc": float(last["val_accuracy"]),
        "reported_latency_ms": met.get("inference_latency_ms_per_image", np.nan),
    })

df = pd.DataFrame(rows)
df["gen_gap_pp"] = (df["final_train_acc"] - df["final_val_acc"]) * 100   # overfitting signal
df["eff_param"] = df["test_top1"] * 100 / (df["total_params"] / 1e6)     # Top-1 % per M-param
df["eff_mem"] = df["test_top1"] * 100 / df["weights_mib"]                # Top-1 % per MiB
df = df.sort_values("test_top1").reset_index(drop=True)                  # ascending accuracy

df.to_csv(COMPARISON / "master_comparison.csv", index=False)
df[["model", "total_params", "weights_mib", "reported_size_mb",
    "test_top1", "test_top5", "macro_f1", "gen_gap_pp", "eff_param"]]

## 7.2 Master comparison table

Emitted as Markdown so it can be pasted directly into Section 7 of the report.

In [ ]:
order = df.sort_values("test_top1", ascending=False)

pct = lambda v: "—" if pd.isna(v) else f"{v * 100:.2f}%"
num = lambda v: "—" if pd.isna(v) else f"{v:,.0f}"
f4  = lambda v: "—" if pd.isna(v) else f"{v:.4f}"
f2  = lambda v: "—" if pd.isna(v) else f"{v:.2f}"

lines = ["| Metric | " + " | ".join(order["model"]) + " |",
         "| :--- | " + " | ".join(["---:"] * len(order)) + " |"]


def row(label, key, fmt):
    lines.append(f"| **{label}** | " + " | ".join(fmt(v) for v in order[key]) + " |")


row("Scaling philosophy",      "scaling",          str)
row("Total parameters",        "total_params",     num)
row("Trainable parameters",    "trainable_params", num)
row("Float32 weights (MiB)",   "weights_mib",      f2)
row("Training wall-clock (s)", "train_time_s",     lambda v: f"{v:,.0f}")
row("Epochs",                  "epochs",           num)
row("Validation Top-1",        "val_top1",         pct)
row("Validation Top-5",        "val_top5",         pct)
row("Test Top-1",              "test_top1",        pct)
row("Test Top-5",              "test_top5",        pct)
row("Test macro precision",    "macro_precision",  f4)
row("Test macro recall",       "macro_recall",     f4)
row("Test macro F1",           "macro_f1",         f4)
row("Test loss",               "test_loss",        f4)
row("Train-val gap (pp)",      "gen_gap_pp",       f2)
row("Top-1 % per M-param",     "eff_param",        f2)
row("Top-1 % per MiB",         "eff_mem",          f2)

table_md = "\n".join(lines) + "\n"
(COMPARISON / "master_comparison.md").write_text(table_md)
print(table_md)

## 7.3 Comparative learning curves

Both panels share one y-scale each — **never a dual axis**. Validation is solid, training
translucent, so the vertical distance between a model's two lines reads directly as its
generalization gap.

In [ ]:
def place_labels(ax, items, x, min_gap_frac=0.055):
    """Nudge end-of-line labels apart so they never collide."""
    lo, hi = ax.get_ylim()
    gap = (hi - lo) * min_gap_frac
    items = sorted(items, key=lambda t: t[0])
    ys = [t[0] for t in items]
    for i in range(1, len(ys)):
        ys[i] = max(ys[i], ys[i - 1] + gap)
    for i in range(len(ys) - 2, -1, -1):
        if ys[i + 1] > hi - gap * 0.5:
            ys[i + 1] = min(ys[i + 1], hi - gap * 0.5)
            ys[i] = min(ys[i], ys[i + 1] - gap)
    for (_, text, _), y in zip(items, ys):
        ax.annotate(text, (x, y), xytext=(6, 0), textcoords="offset points",
                    color=INK_2, fontsize=7.5, va="center", annotation_clip=False)


fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
plot_order = list(df["key"])

for ax, (tr, va, title, ylab) in zip(axes, [
    ("loss", "val_loss", "Cross-entropy loss", "Loss"),
    ("accuracy", "val_accuracy", "Top-1 accuracy", "Accuracy"),
]):
    labels = []
    for m in plot_order:
        h = histories[m]
        e = h["epoch"] + 1                         # histories are 0-indexed
        ax.plot(e, h[tr], color=COLOR[m], lw=2, alpha=0.3, zorder=2)      # training
        ax.plot(e, h[va], color=COLOR[m], lw=2, zorder=3, label=DISPLAY[m])  # validation
        labels.append((float(h[va].iloc[-1]), DISPLAY[m], COLOR[m]))
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylab)
    ax.grid(axis="y", lw=0.8)
    ax.set_axisbelow(True)
    ax.set_xlim(0.5, 20.5)
    ax.set_xticks([1, 5, 10, 15, 20])
    if tr == "accuracy":
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    place_labels(ax, labels, 20.6)

handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", bbox_to_anchor=(0.5, 0.95),
           ncol=4, columnspacing=2.4, handlelength=1.6)
fig.text(0.5, -0.04,
         "Solid = validation, translucent = training. Labels mark the final validation epoch. "
         "Phase 2 fine-tuning begins at epoch 7 (ResNet-50) and epoch 9 (MobileNetV2, EfficientNetB0).",
         ha="center", fontsize=7.5, color=MUTED)
fig.suptitle("Figure 6 — Comparative learning curves, Food-101 "
             "(identical protocol: 224x224, batch 32, seed 42)",
             fontsize=10.5, color=INK, y=1.06)
fig.subplots_adjust(top=0.80, wspace=0.28)
fig.savefig(FIGURES / "fig6_comparative_learning_curves.png")
plt.show()

## 7.4 Accuracy, complexity and cost

Six panels over the same four categories. Every bar carries its value directly, so no reader has
to estimate a height against a gridline.

In [ ]:
panels = [
    ("test_top1",    "Test Top-1 accuracy",     lambda v: f"{v*100:.2f}%", True),
    ("test_top5",    "Test Top-5 accuracy",     lambda v: f"{v*100:.2f}%", True),
    ("macro_f1",     "Test macro F1",           lambda v: f"{v:.4f}",      False),
    ("total_params", "Total parameters (M)",    lambda v: f"{v/1e6:.2f}M", False),
    ("weights_mib",  "Float32 weights (MiB)",   lambda v: f"{v:.1f}",      False),
    ("train_time_s", "Training wall-clock (h)", lambda v: f"{v/3600:.2f}h", False),
]

fig, axes = plt.subplots(2, 3, figsize=(12, 6.4))
for ax, (key, title, fmt, is_pct) in zip(axes.ravel(), panels):
    vals = df[key].values.astype(float)
    scale = {"total_params": 1e6, "train_time_s": 3600}.get(key, 1)
    bars = ax.bar(range(len(df)), np.nan_to_num(vals / scale, nan=0.0),
                  color=[COLOR[k] for k in df["key"]], width=0.62, zorder=3)
    for b, raw in zip(bars, vals):
        missing = np.isnan(raw)
        ax.annotate("not recorded" if missing else fmt(raw),
                    (b.get_x() + b.get_width() / 2, b.get_height()),
                    xytext=(0, 3), textcoords="offset points", ha="center",
                    fontsize=7 if missing else 7.5,
                    color=MUTED if missing else INK,
                    style="italic" if missing else "normal")
    if is_pct:
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_title(title)
    ax.set_xticks(range(len(df)))
    ax.set_xticklabels(df["model"], fontsize=7.5, rotation=18, ha="right")
    ax.grid(axis="y", lw=0.8)
    ax.set_axisbelow(True)
    ax.margins(y=0.18)

fig.suptitle("Figure 7 — Accuracy, complexity and cost across the four architectures",
             fontsize=10.5, color=INK, y=1.0)
fig.text(0.5, -0.02,
         "Weight footprints are computed uniformly as total_params x 4 bytes, the basis Keras "
         "reports. Inference latency is excluded: see Section 8.4.3.",
         ha="center", fontsize=7.5, color=MUTED)
fig.tight_layout()
fig.savefig(FIGURES / "fig7_comparative_metrics.png")
plt.show()

## 7.5 Accuracy against complexity

The central result of the benchmark. A log parameter axis spans the 35x range from Custom CNN to
ResNet-50; every point is direct-labelled, so identity never depends on colour alone.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.8))

for _, r in df.iterrows():
    ax.scatter(r["total_params"] / 1e6, r["test_top1"] * 100, s=170,
               color=COLOR[r["key"]], zorder=3, edgecolor="white", linewidth=2)
    ax.annotate(f"{r['model']}\n{r['test_top1']*100:.2f}% · {r['total_params']/1e6:.2f}M",
                (r["total_params"] / 1e6, r["test_top1"] * 100),
                xytext=(0, 16), textcoords="offset points",
                ha="center", fontsize=7.5, color=INK_2, linespacing=1.35)

eb = df[df["key"] == "efficientnetb0"].iloc[0]
rn = df[df["key"] == "resnet50"].iloc[0]
ax.annotate("", xy=(eb["total_params"] / 1e6, eb["test_top1"] * 100),
            xytext=(rn["total_params"] / 1e6, rn["test_top1"] * 100),
            arrowprops=dict(arrowstyle="->", color=MUTED, lw=1.2,
                            linestyle=(0, (4, 3)), shrinkA=9, shrinkB=9))
ax.annotate(f"{rn['total_params'] / eb['total_params']:.2f}x fewer parameters,\n"
            f"+{(eb['test_top1'] - rn['test_top1']) * 100:.2f} pp accuracy",
            xy=(11.5, 76.5), ha="center", fontsize=7.5, color=INK_2, linespacing=1.35)

ax.set_xlabel("Total parameters (millions, log scale)")
ax.set_ylabel("Test Top-1 accuracy")
ax.set_xscale("log")
ax.set_xlim(0.45, 45)
ax.set_ylim(56, 84)
ax.set_xticks([0.5, 1, 2, 5, 10, 25])
ax.set_xticklabels(["0.5", "1", "2", "5", "10", "25"])
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0f}%")
ax.grid(lw=0.8)
ax.set_axisbelow(True)
ax.set_title("Figure 8 — Accuracy against model complexity: EfficientNetB0 dominates ResNet-50")
fig.savefig(FIGURES / "fig8_accuracy_vs_parameters.png")
plt.show()

## 7.6 Shared error clusters

If the hardest classes were an artefact of one architecture, each model would struggle with a
different set. Ranking all 101 classes by mean F1 across the four models tests that directly, and
supplies the cross-architectural evidence for **Hypothesis 4**.

In [ ]:
per_class = pd.DataFrame({
    DISPLAY[m]: {k: v["f1-score"] for k, v in reports[m].items()
                 if isinstance(v, dict) and "f1-score" in v
                 and k not in ("macro avg", "weighted avg")}
    for m in MODELS
})
per_class["mean_f1"] = per_class.mean(axis=1)
hardest = per_class.nsmallest(10, "mean_f1")

per_class.sort_values("mean_f1").to_csv(COMPARISON / "per_class_f1_all_models.csv")
hardest.to_csv(COMPARISON / "hardest_classes.csv")

fig, ax = plt.subplots(figsize=(8.4, 5.2))
y = np.arange(len(hardest))

for i, (_, r) in enumerate(hardest.iterrows()):     # connector first, dots on top
    span = r[[DISPLAY[m] for m in MODELS]]
    ax.plot([span.min(), span.max()], [i, i], color=GRID, lw=2, zorder=1)
for m in MODELS:
    ax.scatter(hardest[DISPLAY[m]], y, s=90, color=COLOR[m], label=DISPLAY[m],
               zorder=3, edgecolor="white", linewidth=1.6)

ax.set_yticks(y)
ax.set_yticklabels([c.replace("_", " ") for c in hardest.index], fontsize=8.5, color=INK_2)
ax.invert_yaxis()
ax.set_xlabel("Per-class F1 score (test split)")
ax.set_xlim(0.25, 0.66)
ax.grid(axis="x", lw=0.8)
ax.set_axisbelow(True)
ax.set_title("Figure 9 — The ten hardest classes are the same for every architecture", pad=34)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.015), ncol=4,
          columnspacing=2.0, handletextpad=0.4)
fig.text(0.5, -0.03,
         "Ranked by mean F1 across all four models. Compound scaling lifts the whole curve "
         "but does not change which classes are hard (Hypothesis 4).",
         ha="center", fontsize=7.5, color=MUTED)
fig.savefig(FIGURES / "fig9_hardest_classes.png")
plt.show()

print("Hardest ten (mean F1 across all four architectures):")
print(hardest.round(4).to_string())
print("\nEasiest five:")
print(per_class.nlargest(5, "mean_f1").round(4).to_string())

## 7.7 Inference latency — why it is not in the tables above

The four `metrics.json` files each report an `inference_latency_ms_per_image`, but they were
**captured under different protocols** and are not mutually comparable. The symptom is plain:
MobileNetV2, the smallest pretrained model in the benchmark, reports 167.29 ms/image against
ResNet-50's 10.98 ms/image. That ordering is physically impossible and is a measurement artefact,
not a result. EfficientNetB0 is the only model that logged both figures — 13.43 ms/image batched
against 342.26 ms/image at batch size 1 — which shows the spread a protocol change alone produces.

Publishing those four numbers side by side would appear to **refute Hypothesis 2 (edge efficiency)**
on the strength of an instrumentation error, so latency is excluded from Figures 7-8 and from the
master table until it is re-measured.

The cell below re-profiles all four models under one protocol: same batch size, warm-up iterations
discarded, mean over 1,000 test images, same GPU in the same session. It needs the trained weights,
which are excluded from Git by `.gitignore`, so it must run in Colab with the Drive checkpoint
directory mounted. It is guarded and does **not** run by default.

In [ ]:
# Uniform latency re-profiling. Requires a GPU, the dataset and the trained weights.
# Set RUN_LATENCY_PROFILE = True in Colab with Drive mounted; leave False elsewhere.

RUN_LATENCY_PROFILE = False

CHECKPOINTS = {                      # adjust to wherever each member's weights live
    "custom_cnn":     "/content/drive/MyDrive/food101_checkpoints/custom_cnn_best.keras",
    "mobilenetv2":    "/content/drive/MyDrive/food101_checkpoints/mobilenetv2_best.keras",
    "resnet50":       "/content/drive/MyDrive/food101_checkpoints/resnet50_best.keras",
    "efficientnetb0": "/content/drive/MyDrive/food101_checkpoints/efficientnetb0_best.keras",
}

WARMUP_BATCHES = 10      # discarded: first calls pay kernel compilation and cuDNN autotuning
TIMED_IMAGES = 1000

if RUN_LATENCY_PROFILE:
    import time
    import tensorflow as tf
    from src.preprocessing.data_loader import get_food101_datasets

    latency = {}
    for m in MODELS:
        _, _, test_ds = get_food101_datasets(
            data_dir="data/raw/food-101", splits_dir="data/splits",
            model_type=m, image_size=(224, 224), batch_size=32,
        )
        model = tf.keras.models.load_model(CHECKPOINTS[m], compile=False)

        for batch, _ in test_ds.take(WARMUP_BATCHES):     # warm-up, not timed
            model.predict_on_batch(batch)

        n_batches = int(np.ceil(TIMED_IMAGES / 32))
        seen, start = 0, time.perf_counter()
        for batch, _ in test_ds.take(n_batches):
            model.predict_on_batch(batch)
            seen += int(batch.shape[0])
        batched_ms = (time.perf_counter() - start) * 1000 / seen

        single = next(iter(test_ds.unbatch().batch(1).take(100)))[0]
        for _ in range(WARMUP_BATCHES):
            model.predict_on_batch(single)
        start = time.perf_counter()
        for _ in range(100):
            model.predict_on_batch(single)
        single_ms = (time.perf_counter() - start) * 1000 / 100

        latency[m] = {"batched_ms_per_image": round(batched_ms, 3),
                      "batch1_ms_per_image": round(single_ms, 3),
                      "images_timed": seen}
        print(f"{DISPLAY[m]:16s} batched {batched_ms:7.2f} ms  |  batch-1 {single_ms:7.2f} ms")
        del model
        tf.keras.backend.clear_session()

    payload = {"protocol": {"batch_size": 32, "warmup_batches": WARMUP_BATCHES,
                            "timed_images": TIMED_IMAGES, "hardware": "Google Colab NVIDIA T4"},
               "results": latency}
    (COMPARISON / "latency_uniform.json").write_text(json.dumps(payload, indent=2))
    print(f"\n[OK] Wrote {COMPARISON / 'latency_uniform.json'}")
else:
    print("[SKIPPED] RUN_LATENCY_PROFILE is False.")
    print("Reported (non-comparable) values currently in metrics.json:")
    for _, r in df.iterrows():
        print(f"  {r['model']:16s} {r['reported_latency_ms']:>8.2f} ms/image")

## 7.8 Findings

1. **EfficientNetB0 Pareto-dominates ResNet-50.** Higher Top-1 (77.26% vs 73.54%), Top-5
   (94.16% vs 92.02%) and macro F1 (0.7718 vs 0.7355), at 5.69x fewer parameters, 5.69x smaller
   weights and 20.2% less training wall-clock. There is no axis on which ResNet-50 is preferable,
   so depth-only scaling is not justified for this task at this compute budget. **Hypothesis 3 is
   confirmed.**

2. **Transfer learning beats training from scratch decisively.** Every pretrained backbone clears
   the Custom CNN baseline by 7.00 to 15.57 pp of test Top-1. **Hypothesis 1 is confirmed.**

3. **Generalization behaviour separates the architectures more sharply than accuracy does.** Final
   train-validation gaps are 0.03 pp (Custom CNN, underfitting), 1.47 pp (EfficientNetB0, still
   underfitting when its epoch budget ran out), 9.62 pp (MobileNetV2) and 13.06 pp (ResNet-50,
   clearly overfitting — its validation loss begins rising around epoch 15 while training loss
   keeps falling). ResNet-50's extra capacity is being spent memorising the training split.

4. **Test accuracy exceeds validation accuracy for all four models** by 3.96 to 4.57 pp. A property
   shared by four independently trained architectures belongs to the dataset, not to any model: the
   validation split inherits the ~20% web-crawl label noise of the Food-101 training partition,
   whereas the test split was manually cleaned by Bossard et al. (2014).

5. **The error space is a property of the data, not the architecture.** The ten hardest classes are
   essentially identical across all four models, and `steak` is the single worst class for every one
   of them. Compound scaling lifts the whole curve — each hard class gains 0.11-0.16 F1 from
   MobileNetV2 to EfficientNetB0 — without changing which classes are hard. **Hypothesis 4 is
   confirmed.** Further gains will not come from a better backbone; they require higher input
   resolution or hierarchical classification.

6. **Hypothesis 2 (edge efficiency) cannot be settled from the committed artifacts.** MobileNetV2
   does hold the smallest parameter count (2.39M) and the best accuracy-per-parameter ratio among
   the pretrained models (28.77 vs 18.47 Top-1 % per M-param), but the latency evidence is
   unusable until Section 7.7 is re-run under one protocol.